# 🔊 Dorina HiFi-GAN V1: Sıfırdan Vokoder Eğitimi (Kaggle GPU)
Bu notebook, 2.440 adet gerçek Türkçe kadın sesiyle **HiFi-GAN V1 Nöral Vokoderini sıfırdan (From Scratch)** eğitir.

**GPU:** Sağ menüden `Settings -> Accelerator -> GPU T4 x2` veya `P100` seçin.

In [ ]:
!pip install -q soundfile torchaudio pandas pyarrow matplotlib

In [ ]:
# 1. Standart Mel-Spektrogram Modülü (22.050 Hz Saf Standart)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
import os, random, io
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display

class MelConfig:
    sample_rate: int = 22050
    n_fft: int = 1024
    win_length: int = 1024
    hop_length: int = 256
    n_mels: int = 80
    f_min: float = 0.0
    f_max: float = 8000.0

class MelSpectrogramExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_fft=1024, win_length=1024, hop_length=256,
            f_min=0.0, f_max=8000.0, n_mels=80, power=1.0, normalized=False, center=False
        )
    def forward(self, wav):
        if wav.dim() == 1: wav = wav.unsqueeze(0)
        mel = self.mel_transform(wav)
        return torch.log(torch.clamp(mel, min=1e-5))

mel_fn = MelSpectrogramExtractor().cuda()
print("Mel modülü hazır!")

In [ ]:
# 2. HiFi-GAN Mimarisi (Generator, MPD, MSD, Losses)
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dilations=[1, 3, 5]):
        super().__init__()
        self.c1 = nn.ModuleList([nn.Conv1d(channels, channels, kernel_size, d, (kernel_size*d-d)//2) for d in dilations])
        self.c2 = nn.ModuleList([nn.Conv1d(channels, channels, kernel_size, 1, (kernel_size-1)//2) for _ in dilations])
    def forward(self, x):
        for c1, c2 in zip(self.c1, self.c2):
            res = x
            x = F.leaky_relu(x, 0.1)
            x = c1(x)
            x = F.leaky_relu(x, 0.1)
            x = c2(x) + res
        return x

class HiFiGANGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_pre = nn.Conv1d(80, 512, 7, 1, 3)
        self.ups = nn.ModuleList([
            nn.ConvTranspose1d(512, 256, 16, 8, 4),
            nn.ConvTranspose1d(256, 128, 16, 8, 4),
            nn.ConvTranspose1d(128, 64, 4, 2, 1),
            nn.ConvTranspose1d(64, 32, 4, 2, 1),
        ])
        self.res = nn.ModuleList([
            ResBlock(256, 3), ResBlock(256, 7), ResBlock(256, 11),
            ResBlock(128, 3), ResBlock(128, 7), ResBlock(128, 11),
            ResBlock(64, 3), ResBlock(64, 7), ResBlock(64, 11),
            ResBlock(32, 3), ResBlock(32, 7), ResBlock(32, 11),
        ])
        self.conv_post = nn.Conv1d(32, 1, 7, 1, 3)

    def forward(self, x):
        x = self.conv_pre(x)
        r_idx = 0
        for up in self.ups:
            x = F.leaky_relu(x, 0.1)
            x = up(x)
            xs = (self.res[r_idx](x) + self.res[r_idx+1](x) + self.res[r_idx+2](x)) / 3.0
            r_idx += 3
            x = xs
        x = F.leaky_relu(x, 0.1)
        return torch.tanh(self.conv_post(x)).squeeze(1)

class DiscriminatorP(nn.Module):
    def __init__(self, period):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            nn.Conv2d(1, 32, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(32, 128, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(128, 512, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(512, 1024, (5, 1), (3, 1), padding=(2, 0)),
            nn.Conv2d(1024, 1024, (5, 1), 1, padding=(2, 0)),
        ])
        self.post = nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0))
    def forward(self, x):
        b, t = x.shape
        if t % self.period != 0:
            x = F.pad(x, (0, self.period - (t % self.period)), 'reflect')
        x = x.view(b, 1, -1, self.period)
        fmap = []
        for c in self.convs:
            x = F.leaky_relu(c(x), 0.1); fmap.append(x)
        return torch.flatten(self.post(x), 1, -1), fmap

class MPD(nn.Module):
    def __init__(self):
        super().__init__()
        self.discs = nn.ModuleList([DiscriminatorP(p) for p in [2, 3, 5, 7, 11]])
    def forward(self, y, y_g):
        r_outs, g_outs, f_r, f_g = [], [], [], []
        for d in self.discs:
            ro, fr = d(y); go, fg = d(y_g)
            r_outs.append(ro); g_outs.append(go); f_r.append(fr); f_g.append(fg)
        return r_outs, g_outs, f_r, f_g

print("HiFi-GAN Mimarisi hazır!")

In [ ]:
# 3. Veri Kümesini İndir (2.440 Türkçe Kadın Sesi)
os.makedirs("wavs", exist_ok=True)
repos = [f"falan42/Turkish-Bentropi-tts-dataset-part-{i}" for i in range(2, 12)]
total = 0
for r in repos:
    url = f"https://huggingface.co/datasets/{r}/resolve/main/data/train-00000-of-00001.parquet"
    try:
        df = pd.read_parquet(url)
        for _, row in df.iterrows():
            ab = row['audio']['bytes']
            w, sr = sf.read(io.BytesIO(ab))
            t_w = torch.tensor(w, dtype=torch.float32)
            if t_w.dim() > 1: t_w = t_w.t()[:1]
            else: t_w = t_w.unsqueeze(0)
            if sr != 22050:
                t_w = torchaudio.transforms.Resample(sr, 22050)(t_w)
            sf.write(f"wavs/{total:05d}.wav", t_w.squeeze(0).numpy(), 22050)
            total += 1
        print(f"[+] {r} indirildi. Toplam ses: {total}")
    except Exception as e:
        pass
print(f"Tüm veri kümesi tamamlandı! Toplam: {total} ses dosyası.")

In [ ]:
# 4. Vokoder Eğitim Döngüsü (LSGAN + Mel Loss + Feature Matching)
generator = HiFiGANGenerator().cuda()
mpd = MPD().cuda()

opt_g = torch.optim.AdamW(generator.parameters(), lr=2e-4, betas=(0.8, 0.99))
opt_d = torch.optim.AdamW(mpd.parameters(), lr=2e-4, betas=(0.8, 0.99))

wav_files = [os.path.join("wavs", f) for f in os.listdir("wavs") if f.endswith(".wav")]
SEGMENT_SIZE = 8192 # Claude önerisi: 8192 örnek

print("[*] Eğitim başlatılıyor...")
for step in range(1, 10001):
    # Rastgele bir WAV seç ve 8192 örnek kes
    fn = random.choice(wav_files)
    data, _ = sf.read(fn)
    if len(data) < SEGMENT_SIZE + 1000: continue
    start = random.randint(0, len(data) - SEGMENT_SIZE)
    chunk = torch.tensor(data[start:start+SEGMENT_SIZE], dtype=torch.float32).unsqueeze(0).cuda()
    
    # Mel çıkar
    with torch.no_grad():
        mel = mel_fn(chunk)
        
    # Generator üretim
    y_g = generator(mel)
    min_len = min(chunk.shape[-1], y_g.shape[-1])
    chunk = chunk[:, :min_len]
    y_g = y_g[:, :min_len]
    
    # DISCRIMINATOR ADIMI
    opt_d.zero_grad()
    r_outs, g_outs, _, _ = mpd(chunk, y_g.detach())
    loss_d = sum(torch.mean((1-ro)**2) + torch.mean(go**2) for ro, go in zip(r_outs, g_outs))
    loss_d.backward()
    opt_d.step()
    
    # GENERATOR ADIMI (L_G = L_adv + 2*L_FM + 45*L_mel)
    opt_g.zero_grad()
    _, g_outs, f_r, f_g = mpd(chunk, y_g)
    loss_adv = sum(torch.mean((1-go)**2) for go in g_outs)
    loss_fm = sum(torch.mean(torch.abs(r-g)) for fr, fg in zip(f_r, f_g) for r, g in zip(fr, fg)) * 2.0
    loss_mel = F.l1_loss(mel_fn(y_g), mel_fn(chunk)) * 45.0
    
    loss_g = loss_adv + loss_fm + loss_mel
    loss_g.backward()
    opt_g.step()
    
    if step % 200 == 0 or step == 1:
        print(f"Step [{step}/10000] - Loss G: {loss_g.item():.4f} (Mel: {loss_mel.item():.4f}) | Loss D: {loss_d.item():.4f}")

torch.save(generator.state_dict(), "dorina_hifigan_v1.pt")
print("[OK] Dorina Vokoder başarıyla eğitildi ve kaydedildi!")

In [ ]:
# 5. Vokoderi Doğrula (GT Mel -> Senin Vokoderin -> WAV Dinle!)
generator.eval()
with torch.no_grad():
    test_data, _ = sf.read(wav_files[0])
    t_tensor = torch.tensor(test_data, dtype=torch.float32).unsqueeze(0).cuda()
    gt_mel = mel_fn(t_tensor)
    rec_wav = generator(gt_mel).cpu().squeeze().numpy()
    sf.write("test_our_vocoder.wav", rec_wav, 22050)
    print("Doğrulama sesi hazır! Aşağıdan dinleyin:")
    display(Audio("test_our_vocoder.wav", rate=22050))
